# Modul 6: Gedächtnis & Kontext

**Agent Systems: Vom Konzept zum eigenen Orchestrator**

In diesem Notebook baust du ein 3-stufiges Memory-System mit Retrieval.

🎯 **Lernziel:** Die 3 Memory-Stufen (Session → Tages-File → Langzeit-Summary) implementieren und Infos über Sessions hinweg abrufen.

---

## Die 3 Stufen des Agent-Gedächtnisses

```
┌──────────────────────────────────────────┐
│  Stufe 3: LANGZEIT (MEMORY.md)          │
│  → Kompakte Summary, überlebt Wochen     │
├──────────────────────────────────────────┤
│  Stufe 2: TAGES-FILE (memory/YYYY.md)   │
│  → Tagesnotizen, überlebt Sessions       │
├──────────────────────────────────────────┤
│  Stufe 1: SESSION (Kontext-Fenster)     │
│  → Aktuelle Konversation, flüchtig       │
└──────────────────────────────────────────┘
```

Park et al. (2023, "Generative Agents"): **Memory-Stream + Retrieval + Reflection** erzeugen emergentes Verhalten.

In [ ]:
# 3-stufiges Memory-System

from datetime import datetime


class SessionMemory:
    """Stufe 1: Flüchtiges Gedächtnis der aktuellen Konversation."""
    def __init__(self, max_turns=20):
        self.history = []
        self.max_turns = max_turns

    def add(self, role, content):
        self.history.append({'role': role, 'content': content, 'time': datetime.now().isoformat()})
        if len(self.history) > self.max_turns:
            self.history = self.history[-self.max_turns:]

    def search(self, keyword):
        return [m for m in self.history if keyword.lower() in m['content'].lower()]


class DailyMemory:
    """Stufe 2: Tagesnotizen die Sessions überleben."""
    def __init__(self):
        self.files = {}  # Datum -> Liste von Einträgen

    def save(self, category, note, date=None):
        d = date or datetime.now().strftime('%Y-%m-%d')
        if d not in self.files:
            self.files[d] = []
        entry = f'{datetime.now().strftime("%H:%M")} [{category}] {note}'
        self.files[d].append(entry)
        print(f'  Notiz ({d}): {entry}')

    def search(self, keyword):
        results = []
        for date, entries in self.files.items():
            for e in entries:
                if keyword.lower() in e.lower():
                    results.append({'date': date, 'entry': e})
        return results


class LongTermMemory:
    """Stufe 3: Kompakte Langzeit-Summary."""
    def __init__(self, max_chars=2000):
        self.facts = {}
        self.max_chars = max_chars

    def store(self, key, value):
        self.facts[key] = value
        # Limit einhalten
        total = sum(len(k) + len(str(v)) for k, v in self.facts.items())
        while total > self.max_chars and self.facts:
            oldest = next(iter(self.facts))
            del self.facts[oldest]
            total = sum(len(k) + len(str(v)) for k, v in self.facts.items())
        print(f'  Langzeit: {key} = {value}')

    def search(self, keyword):
        return {k: v for k, v in self.facts.items()
                if keyword.lower() in k.lower() or keyword.lower() in str(v).lower()}

    def summary(self):
        return '\n'.join(f'- {k}: {v}' for k, v in self.facts.items()) or '(leer)'


class AgentMemory:
    """Vereint alle 3 Stufen mit Retrieval."""
    def __init__(self):
        self.session = SessionMemory()
        self.daily = DailyMemory()
        self.longterm = LongTermMemory()

    def remember(self, keyword):
        """Durchsucht alle 3 Stufen."""
        print(f'\n Suche: "{keyword}"')
        s = self.session.search(keyword)
        d = self.daily.search(keyword)
        lt = self.longterm.search(keyword)
        if s:  print(f'  Session: {len(s)} Treffer')
        if d:  print(f'  Tagesnotizen: {len(d)} Treffer')
        if lt: print(f'  Langzeit: {len(lt)} Treffer')
        if not s and not d and not lt:
            print(f'  Nichts gefunden.')
        return {'session': s, 'daily': d, 'longterm': lt}


# === Demo ===
print('=== Memory-System Demo ===\n')
mem = AgentMemory()

# Stufe 1: Session
mem.session.add('user', 'Mein Lieblingsbuch ist Dune.')
mem.session.add('assistant', 'Gemerkt! Du magst Dune.')
mem.session.add('user', 'Ich arbeite an einem Python-Projekt.')
print('Session geladen.\n')

# Stufe 2: Tagesnotizen
mem.daily.save('IDEE', 'Agent mit 4 Schichten bauen', '2026-03-24')
mem.daily.save('TODO', 'Python-Projekt fertigstellen', '2026-03-24')
mem.daily.save('LEARNING', 'ReAct-Loop verbessert Completion um 20-30%', '2026-03-23')
print()

# Stufe 3: Langzeit
mem.longterm.store('lieblingsbuch', 'Dune von Frank Herbert')
mem.longterm.store('sprache', 'Deutsch')
mem.longterm.store('timezone', 'Europe/Berlin')
print()

# Retrieval
mem.remember('Python')
mem.remember('Dune')
mem.remember('Katze')

In [ ]:
# Konsolidierung: Tagesnotizen -> Langzeit

def consolidate(memory, date_str):
    """Konsolidiert Tagesnotizen in Langzeit-Fakten."""
    entries = memory.daily.files.get(date_str, [])
    if not entries:
        print(f'Keine Eintraege fuer {date_str}')
        return
    print(f'\nKonsolidiere {date_str} ({len(entries)} Eintraege)...')
    for entry in entries:
        if '[IDEE]' in entry:
            memory.longterm.store(f'idee_{date_str}', entry.split('] ')[1])
        elif '[LEARNING]' in entry:
            memory.longterm.store(f'learning_{date_str}', entry.split('] ')[1])
    print(f'\nLangzeit-Summary:')
    print(memory.longterm.summary())

consolidate(mem, '2026-03-24')
consolidate(mem, '2026-03-23')

In [ ]:
# 🎯 ÜBUNG: Baue eine Relevanz-Ranking-Funktion
#
# Ergebnisse ranken: Session=Prio 3 (aktuellstes), Tagesnotizen=Prio 2, Langzeit=Prio 1.
# Sortiere nach Priorität (höchste zuerst).

def ranked_remember(memory, keyword):
    results = []
    # TODO: Session durchsuchen, Priorität 3
    # for hit in memory.session.search(keyword):
    #     results.append({'source': 'session', 'priority': 3, 'content': hit['content']})

    # TODO: Tagesnotizen durchsuchen, Priorität 2

    # TODO: Langzeit durchsuchen, Priorität 1

    # TODO: Sortieren nach Priorität (absteigend)
    return results

# Test
# for r in ranked_remember(mem, 'Python'):
#     print(f'  [Prio {r["priority"]}] {r["source"]}: {r["content"]}')

In [ ]:
# ✅ LÖSUNG

def ranked_remember(memory, keyword):
    results = []

    for hit in memory.session.search(keyword):
        results.append({'source': 'session', 'priority': 3, 'content': hit['content']})

    for hit in memory.daily.search(keyword):
        results.append({'source': f'daily ({hit["date"]})', 'priority': 2, 'content': hit['entry']})

    for key, val in memory.longterm.search(keyword).items():
        results.append({'source': 'longterm', 'priority': 1, 'content': f'{key}: {val}'})

    results.sort(key=lambda x: x['priority'], reverse=True)
    return results

print('Ranked Search: "Python"')
for r in ranked_remember(mem, 'Python'):
    print(f'  [Prio {r["priority"]}] {r["source"]}: {r["content"]}')

print('\nRanked Search: "Dune"')
for r in ranked_remember(mem, 'Dune'):
    print(f'  [Prio {r["priority"]}] {r["source"]}: {r["content"]}')

## Vergleich: Memory in Frameworks

| Framework | Memory-Ansatz | Persistenz |
|-----------|--------------|------------|
| **OpenClaw** | File-basiert (memory/*.md + MEMORY.md) | Dateisystem |
| **LangChain** | ConversationBufferMemory / VectorStore | In-Memory / DB |
| **MemGPT** | Hierarchisch (Main/Archival) | Paged Memory |
| **AutoGen** | Teachability + Long-term | ChromaDB |

---

### ✅ Self-Check
- [ ] Ich kann die 3 Memory-Stufen benennen und unterscheiden
- [ ] Ich verstehe wie Konsolidierung (Tages → Langzeit) funktioniert
- [ ] Ich kann Retrieval über alle Stufen implementieren

**→ Weiter zu [Modul 7a: Skills nutzen](https://colab.research.google.com/github/planck-lab/agent-systems/blob/main/notebooks/modul7a_skills_nutzen.ipynb)**